# Document Embedding Demo

In [ ]:
# !pip install transformers
# !pip install sentence-transformers
# !pip install langchain-community langchain_huggingface

In [ ]:
DEVICE = "cuda"    # "cpu"

In [ ]:
# Jet engines - Wikipedia
document_1 = """
A jet engine is a type of reaction engine, discharging a fast-moving jet of heated gas (usually air) that generates thrust by jet propulsion. 
While this broad definition may include rocket, water jet, and hybrid propulsion, the term jet engine typically refers to an internal combustion 
air-breathing jet engine such as a turbojet, turbofan, ramjet, pulse jet, or scramjet. In general, jet engines are internal combustion engines.
Air-breathing jet engines typically feature a rotating air compressor powered by a turbine, with the leftover power providing thrust through 
the propelling nozzle—this process is known as the Brayton thermodynamic cycle. Jet aircraft use such engines for long-distance travel. 
Early jet aircraft used turbojet engines that were relatively inefficient for subsonic flight. Most modern subsonic jet aircraft use more 
complex high-bypass turbofan engines. They give higher speed and greater fuel efficiency than piston and propeller aeroengines over long distances.
A few air-breathing engines made for high-speed applications (ramjets and scramjets) use the ram effect of the vehicle's speed instead of 
a mechanical compressor.
"""

# Jet engines - NASA
document_2 = """
All jet engines, which are also called gas turbines, work on the same principle. The engine sucks air in at the front with a fan. 
A compressor raises the pressure of the air. The compressor is made with many blades attached to a shaft. The blades spin at high speed and compress 
or squeeze the air. The compressed air is then sprayed with fuel and an electric spark lights the mixture. The burning gases expand and blast out 
through the nozzle, at the back of the engine. As the jets of gas shoot backward, the engine and the aircraft are thrust forward. 
As the hot air is going to the nozzle, it passes through another group of blades called the turbine. The turbine is attached to the same shaft as 
the compressor. Spinning the turbine causes the compressor to spin.
"""

#### Never ever split text into sentences in such a naive way (like we are about to do) for use in production systems!

In [ ]:
sentences_1 = list(map(lambda s: s.replace("\n", "").strip(), document_1.split(".")))[:-1]
print(len(sentences_1))
sentences_1

In [ ]:
sentences_2 = list(map(lambda s: s.replace("\n", "").strip(), document_2.split(".")))[:-1]
sentences_2

## Using `transformers`

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

In [ ]:
# Mean Pooling - Take attention mask into account for correct averaging

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]    # First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

In [ ]:
# Load model from HuggingFace Hub

tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-mpnet-base-v2")
embedding_model_A = AutoModel.from_pretrained("sentence-transformers/all-mpnet-base-v2").to(DEVICE)

In [ ]:
# Tokenize sentences

encoded_input_1 = tokenizer(sentences_1, padding=True, truncation=True, return_tensors="pt").to(DEVICE)
encoded_input_2 = tokenizer(sentences_2, padding=True, truncation=True, return_tensors="pt").to(DEVICE)

In [ ]:
encoded_input_1["input_ids"].shape

In [ ]:
# encoded_input_1

In [ ]:
# Compute token embeddings

with torch.no_grad():
    model_A_output_1 = embedding_model_A(**encoded_input_1)
    model_A_output_2 = embedding_model_A(**encoded_input_2)

In [ ]:
model_A_output_1[0].shape

In [ ]:
model_A_output_2[0].shape

> 🤔 **wait a second...** Why is the output of a sentence embedding model at the token level? Didn't we learn that sentence embedding models are trained at the sentence level, minimizing the difference between semantically similar sentences and maximizing the difference between semantically different sentences? The model works at the sentence level, so it must produce sentence level embedding, right? Not really. It only means the loss is computed on a sentence-level representation somewhere downstream in the computation graph. The output is still at the single token level, where the embedding weights are "adjusted" such that semantically similar sentences will be "closer" together at high-dimensional space (and vice-versa).

In [ ]:
# Perform pooling

sentence_embeddings_A_1 = mean_pooling(model_A_output_1, encoded_input_1["attention_mask"])
sentence_embeddings_A_2 = mean_pooling(model_A_output_2, encoded_input_2["attention_mask"])

# Normalize embeddings (to unit vectors)

sentence_embeddings_A_1 = F.normalize(sentence_embeddings_A_1, p=2, dim=1)
sentence_embeddings_A_2 = F.normalize(sentence_embeddings_A_2, p=2, dim=1)

In [ ]:
sentence_embeddings_A_1.shape

In [ ]:
sentence_embeddings_A_1.dtype

In [ ]:
# Pooling on senetence embeddings to get document embeddings

document_embedding_A_1 = sentence_embeddings_A_1.mean(dim=0)
document_embedding_A_2 = sentence_embeddings_A_2.mean(dim=0)

In [ ]:
document_embedding_A_1.shape

In [ ]:
document_embedding_A_2.shape

#### Cosine Similarity Score

In [ ]:
from sentence_transformers import util

If we don't normalize, then we need to use the cosine similarity function.

In [ ]:
util.cos_sim(a=document_embedding_A_1, b=document_embedding_A_2)

If we do normalize, then a simple dot-product will give us the exact same result:

In [ ]:
document_embedding_A_1_norm = torch.nn.functional.normalize(document_embedding_A_1, dim=0)
document_embedding_A_2_norm = torch.nn.functional.normalize(document_embedding_A_2, dim=0)

In [ ]:
util.dot_score(document_embedding_A_1_norm, document_embedding_A_2_norm)

## Using `sentence-transformers`

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
# embedding_model_B = SentenceTransformer(model_name_or_path="all-mpnet-base-v2", device=DEVICE)
embedding_model_B = SentenceTransformer(model_name_or_path="all-MiniLM-L6-v2", device=DEVICE)

In [ ]:
# Compute token embeddings

sentence_embeddings_B_1 = embedding_model_B.encode(sentences_1, normalize_embeddings=True, convert_to_tensor=True, device=DEVICE)
sentence_embeddings_B_2 = embedding_model_B.encode(sentences_2, normalize_embeddings=True, convert_to_tensor=True, device=DEVICE)

In [ ]:
sentence_embeddings_B_1.shape

In [ ]:
# Pooling on senetence embeddings to get document embeddings

document_embedding_B_1 = sentence_embeddings_B_1.mean(dim=0)
document_embedding_B_2 = sentence_embeddings_B_2.mean(dim=0)

In [ ]:
util.cos_sim(a=document_embedding_B_1, b=document_embedding_B_2)

## Using `LangChain`

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
model_name = "BAAI/bge-small-en-v1.5"
model_kwargs = {"device": DEVICE}
encode_kwargs = {
    "normalize_embeddings": True,
    "convert_to_tensor": True
}

In [ ]:
embedding_model_C = HuggingFaceEmbeddings(model_name=model_name, model_kwargs=model_kwargs, encode_kwargs=encode_kwargs)

In [ ]:
document_embedding_C_1 = torch.tensor(embedding_model_C.embed_documents(texts=[document_1])[0], dtype=torch.float32)
document_embedding_C_2 = torch.tensor(embedding_model_C.embed_documents(texts=[document_2])[0], dtype=torch.float32)

In [ ]:
document_embedding_C_1.shape

In [ ]:
util.dot_score(document_embedding_C_1, document_embedding_C_2)

<hr/>